# Downstream Analysis

In [ ]:
import os
import anndata as ad
import scanpy as sc
import numpy as np
import pandas as pd
import random
from matplotlib import pyplot as plt

import seaborn as sns
from matplotlib import rcParams
import textwrap
import re
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from tqdm import tqdm
from pqdm.threads import pqdm

import sys
sys.path.insert(0, "/Users/nirreiter/code/pyslingshot")
sys.path.insert(0, "/Users/nirreiter/code/pcurvepy")
from pyslingshot import Slingshot

# from dotenv import load_dotenv
# load_dotenv()
# print(os.environ.get("RB_LAB_DATA_PATH"))

## Load Data

In [ ]:
data_dir = "."
output_dir = "output"
input_file = "annotation_data.h5ad"

DPI = 200
sc.settings.figdir = output_dir
sc.settings.set_figure_params(dpi=50, facecolor="white", figsize=(8, 5))

from matplotlib import colors as mplc
_colors = ["lightgray", "royalblue", "gold"]
cmap = mplc.LinearSegmentedColormap.from_list("gray_blue_yellow", _colors)

SEED = 7
np.random.seed(SEED)
random.seed(SEED)

In [ ]:
adata = ad.read_h5ad(input_file)
adata

In [ ]:
treatment_groups = adata.obs["sample"].unique()
print(treatment_groups)
cell_types = adata.obs["cell_type"].unique().dropna()
print(cell_types)

In [ ]:
sc.pl.umap(adata, color=["cell_type", "scDblFinder.score"])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
adata.obs.groupby("cell_type")["scDblFinder.score"].mean().plot.bar(zorder=3, ax=ax)
plt.grid(visible=True, axis="y")
plt.grid(visible=False, axis="x")

In [ ]:
sc.pl.umap(adata, color=["cell_supertype", "cell_type"], legend_loc="on data")

In [ ]:
adata.obs["cell_type"].unique().to_list()

In [ ]:
adata.obs["cell_supertype"].unique().to_list()

In [ ]:
type_order = [
    'AT1',
    'AT2 I',
    'AT2 II',
    'Club',
    'Goblet',
    'Ciliated',
    
    'Fibroblast',
    'Pericyte / VSMC',
    'gCap',
    'aCap',
    'Erythroid',
    
    'Alveolar Macrophage',
    'Interstitial Macrophage',
    'Monocyte-derived Macrophage',
    'IFN-Activated Macrophage',
    
    'cDC1',
    'cDC2',
    'Migratory cDC',
    'pDC',
    
    'Follicular B',
    'Activated B',
    'Circulating B',
    'Resident Memory B',
    'GC B',
    'Fcrl5+ Atypical B',
    
    'IgA+ Plasma',
    'IgG1+ Plasma',
    'IgG2b+ Plasma',
    'IgG2c+ Plasma',
    'IgM+ Plasma',
    
    'Treg',
    'Tfh',
    'Th17',
    'Naive CD8+ T',
    'Effector CD8+ T',
    'γδ T',
    'NK',
    'ILC2',
    
    'Neutrophil',
]

supertype_order = [
    'AT1',
    'AT2',
    'Club',
    'Goblet',
    'Ciliated',
    
    'Fibroblast',
    'Pericyte / VSMC',
    'Endothelial',
    'Erythroid',
    
    'Macrophage',
    'DC',
    
    'B',
    'Plasma',
    'T',
    'NK',
    'ILC',
    'Neutrophil',
]

for c in adata.obs["cell_type"].unique():
    if c not in type_order:
        print(c)

for c in adata.obs["cell_supertype"].unique():
    if c not in supertype_order:
        print(c)

In [ ]:
def wrap_join(items, sep=" ", width=30) -> str:
    lines = []
    current = items[0]

    for item in items[1:]:
        if len(current) + len(sep) + len(item) > width:
            lines.append(current)
            current = item
        else:
            current += sep + item
    lines.append(current)
    
    return "\n".join(lines)

def title(s: str) -> str:
    return " ".join([s[0].upper() + s[1:] for s in s.replace("_", " ").split(" ")])

def rename(t: str, additional_renaming: dict[str, str] | None = None) -> str:
    if additional_renaming is not None and t in additional_renaming:
        return additional_renaming[t]
    if adata is None or t not in adata.uns["rename_dict"]:
        return t
    return adata.uns["rename_dict"][t]

def make_colorbar(sm: mpl.cm.ScalarMappable, ax: mpl.axes.Axes, label: str, vmin, vmax, vcenter=None): # pyright: ignore[reportAttributeAccessIssue]
    cbar = plt.colorbar(sm, cax=ax)
    cbar.set_label(label, fontsize=14)
    cbar.ax.tick_params(labelsize=10)
    ticks = list(cbar.get_ticks())
    # remove the last tick if it is too close to the maximum value to avoid tick collision
    if vmax - ticks[-2] < (ticks[-2] - ticks[-3]) / 4:
        ticks = ticks[:-2] + [vmax]
    else:
        ticks = ticks[:-1] + [vmax]
    cbar.set_ticks(ticks, labels=[f"{tick:.2f}" for tick in ticks]) # pyright: ignore[reportArgumentType]
    # cbar.ax.set_ylim(vmin, vmax)

def minmax_int_slow_with_zero(data):
    return int(np.floor(min(min(data), 0))), int(np.ceil(max(max(data), 0)))

def umap(
    adata: ad.AnnData, 
    feature: str, 
    cmap: mpl.colors.Colormap | dict = cmap, # pyright: ignore[reportAttributeAccessIssue]
    umap_obsm = None,
    figsize=None,
    legend_loc: str = "on data",
    legend_renaming = None,
    ax = None, 
    side_ax = None,
    vmin = None,
    vmax = None,
    vcenter = None,
    s = None,
    show_grid = False,
    xlim = None,
    ylim = None,
):   
    
    if ax is None:
        fig = plt.figure(figsize=figsize or (10, 5), dpi=DPI)
        gs = mpl.gridspec.GridSpec(1, 2, width_ratios=[1, 0.05]) # pyright: ignore[reportAttributeAccessIssue]
        ax = fig.add_subplot(gs[0, 0])
        if side_ax is None:
            side_ax = fig.add_subplot(gs[0, 1])
    else:
        gs = None
    
    umap_x = adata.obsm[umap_obsm or "X_umap"][:, 0]
    umap_y = adata.obsm[umap_obsm or "X_umap"][:, 1]
    
    if s is None:
        area = (max(umap_x) - min(umap_x)) * (max(umap_y) - min(umap_y))
        s = 1000 / area
    
    groups = None
    data = None
    
    if feature in adata.obs and adata.obs[feature].dtype == "category":
        groups = adata.obs[feature].cat.categories
        if isinstance(cmap, dict):
            colors = groups.map(cmap).to_numpy()
            point_colors = adata.obs[feature].astype(object).map(cmap).to_numpy()
        else:
            colors = adata.uns[feature + "_colors"]# if (feature + "_colors") in adata.uns else sns.color_palette(n_colors=len(groups))
            point_colors = np.array(colors)[adata.obs[feature].cat.codes.values] # pyright: ignore[reportArgumentType, reportCallIssue]
        # shuffle points to prevent 1 sample from appearing on top
        if feature == "sample":
            shuffle_index = np.arange(len(umap_x))
            np.random.shuffle(shuffle_index)
            umap_x = umap_x[shuffle_index] # pyright: ignore[reportArgumentType, reportCallIssue]
            umap_y = umap_y[shuffle_index] # pyright: ignore[reportArgumentType, reportCallIssue]
            point_colors = point_colors[shuffle_index]
        
    else:
        if feature in adata.obs:
            data = adata.obs[feature]
        else:
            data = adata.raw[:, feature].X
        
        if hasattr(data, "to_numpy"):  # pandas Series
            values = data.to_numpy().flatten() # pyright: ignore[reportAttributeAccessIssue]
        elif not hasattr(data, "toarray"):  # dense array
            values = data.flatten()  # pyright: ignore[reportAttributeAccessIssue]
        else:
            values = data.toarray().flatten() # pyright: ignore[reportAttributeAccessIssue]
        
        if len(values) == 0:
            return ax, side_ax, None
        
        if vmin is None:
            vmin = np.nanmin(values)
        if vmax is None:
            vmax = np.nanmax(values)
        
        ax.scatter(
            umap_x,
            umap_y,
            c="lightgrey",
            edgecolors="none",
            s=s,
        )
        
        nonzero = values.A1 > 0 if hasattr(data, "A1") else values > 0 # pyright: ignore[reportAttributeAccessIssue]
        umap_x = umap_x[nonzero]
        umap_y = umap_y[nonzero]
        values = values[nonzero]
        
        # Setup shared ScalarMappable for colorbar
        if vcenter is not None:
            norm = mpl.colors.TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)
        else:
            norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax) # pyright: ignore[reportAttributeAccessIssue]
        sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm) # pyright: ignore[reportAttributeAccessIssue]
        sm.set_array([])
        point_colors = sm.to_rgba(values)
        sort_idx = np.argsort(values)
        
        umap_x = umap_x[sort_idx] # pyright: ignore[reportCallIssue, reportArgumentType]
        umap_y = umap_y[sort_idx] # pyright: ignore[reportCallIssue, reportArgumentType]
        point_colors = point_colors[sort_idx]
    
    ax.scatter(
        umap_x,
        umap_y,
        c=point_colors,
        edgecolors="none",
        s=s,
        alpha=0.7,
    )

    added_text = []
    if groups is not None:
        if legend_loc == "on data" and feature != "sample":
            # Place label at median position of each group
            for idx, cat in enumerate(groups):
                mask = adata.obs[feature] == cat
                median_x = np.median(umap_x[mask])
                median_y = np.median(umap_y[mask])
                added_text.append(ax.text(
                    median_x, # pyright: ignore[reportArgumentType]
                    median_y, # pyright: ignore[reportArgumentType]
                    rename(str(cat), legend_renaming),
                    fontsize=10,
                    weight="bold",
                    color="black",
                    ha="center",
                    va="center",
                    bbox=dict(facecolor="white", edgecolor=colors[idx], boxstyle="round,pad=0.2", alpha=0.7)
                ))
            if side_ax:
                side_ax.axis("off")
        else:
            categories = adata.obs[feature].cat.categories
            colors = adata.uns[feature + "_colors"]
            color_dict = dict(zip(categories, colors))
            handles = [
                plt.Line2D([], [], marker='s', linestyle='None', color=color_dict[cat], # pyright: ignore[reportPrivateImportUsage]
                        label=rename(cat, legend_renaming), markersize=6)
                for cat in sorted(groups, key=int if feature == "cluster" else None)
            ]
            if side_ax:
                side_ax.legend(
                    handles=handles,
                    loc='center left',
                    # frameon=False,
                    title=title(rename(feature, legend_renaming)),
                    fontsize=10,
                    title_fontsize=13,
                )
                side_ax.set_facecolor('none')  # No background color
                side_ax.set_xticks([])
                side_ax.set_yticks([])
                for spine in side_ax.spines.values():
                    spine.set_visible(False)
                side_ax.patch.set_alpha(0)
    else:
        if side_ax:
            # Create the colorbar manually
            cbar = plt.colorbar(sm, cax=side_ax)
            
            # Set label and font size
            cbar.set_label(f"{feature} Expression", fontsize=14)
            cbar.ax.tick_params(labelsize=10)
            
            # Force setting ticks if missing
            if cbar.ax.get_yticks().size == 0:
                # Example: set 5 ticks evenly spaced
                if vmin is not None and vmax is not None:
                    ticks = np.linspace(vmin, vmax, 5)
                else:
                    print(f"WARNING! vmin or vmax were None when trying to draw colorbar. vmin: {vmin}, vmax: {vmax}")
                cbar.set_ticks(ticks) # pyright: ignore[reportArgumentType]
                cbar.ax.set_yticklabels([f"{tick:.2g}" for tick in ticks])

    ax.set_title(title(rename(feature)))
    ax.set_xlabel(rename("UMAP 1"))
    ax.set_ylabel(rename("UMAP 2"))
    ax.set_xticks([])
    ax.set_yticks([])
    # ax.grid(show_grid)
    if show_grid:
        for x in range(*minmax_int_slow_with_zero(adata.obsm["X_umap"][:, 0])):
            ax.axvline(x=x, color="grey")
        for y in range(*minmax_int_slow_with_zero(adata.obsm["X_umap"][:, 1])):
            ax.axhline(y=y, color="grey")
        ax.axvline(x=0, color="black")
        ax.axhline(y=0, color="black")
    
    if xlim is not None:
        ax.set_xlim(*xlim)
    if ylim is not None:
        ax.set_ylim(*ylim)
    
    plt.tight_layout()
    return ax, side_ax, added_text

def multiple_umap(
    adata: ad.AnnData | list[ad.AnnData], 
    features: list[str], 
    cmap: mpl.colors.Colormap = None, # pyright: ignore[reportAttributeAccessIssue]
    umap_obsm = None,
    legend_loc = "on data",
):
    if isinstance(adata, ad.AnnData):
        adata = [adata]
    
    ngraphs = len(adata) * len(features)
    w = 2 if ngraphs > 1 else 1
    h = int(np.ceil(ngraphs / 2))
    fig, axes = plt.subplots(
        h, 2 if w == 1 else 5, figsize=(11 if w == 1 else 25, 10 * h), 
        gridspec_kw={
            'width_ratios': [20, 1] if w == 1 else [20, 1, 5, 20, 1],
        },
        dpi=DPI,
    )
    if h == 1:
        axes = axes.reshape(1, -1)
    if w == 1 and h == 1:
        axes = axes.reshape(1, 2)
    
    added_text = []
    
    for y in range(h):
        for x in range(w):
            index = y * w + x
            ax = axes[y, x*3]
            side_ax = axes[y, x*3 + 1]
            if index < ngraphs:
                added_text.append(umap(
                    adata[index // len(features)],
                    features[index % len(features)], 
                    cmap,
                    umap_obsm,
                    legend_loc, 
                    ax = ax, 
                    side_ax = side_ax,
                )[2])
            else:
                axes[y, x].set_facecolor('none')  # No background color
                axes[y, x].set_xticks([])
                axes[y, x].set_yticks([])
                for spine in axes[y, x].spines.values():
                    spine.set_visible(False)
                axes[y, x].patch.set_alpha(0)
    
    if w > 1:
        for y in range(h):
            axes[y, 2].set_facecolor('none')  # No background color
            axes[y, 2].set_xticks([])
            axes[y, 2].set_yticks([])
            for spine in axes[y, 2].spines.values():
                spine.set_visible(False)
            axes[y, 2].patch.set_alpha(0)
    
    plt.subplots_adjust(
        left=0.08,
        right=0.92,  
        top=0.95,
        bottom=0.05,
        hspace=0.2,
        wspace=0.1
    )
    
    return fig
            

def umap_split(
    adata, 
    feature,
    group_key,
    umap_obsm = None,
    legend_kws = {},
    legend_order = None,
    figsize = None,
    cmap = cmap,
    **kwargs
):
    fig = plt.figure(figsize=figsize or (10, 8), dpi=DPI)  # wider to fit legend or colorbar
    
    groups = adata.obs[group_key].unique().tolist()
    # adatas = [adata[adata.obs[group_key] == t] for t in groups]
    # if isinstance(features, str):
    #     features = [features]
    # return multiple_umap(adatas, features, **kwargs)
    
    w = 2
    h = max(1, int(np.ceil(len(groups) / w)))

    # Prepare figure with 2x2 plots and one extra row (for colorbar/legend)
    gs = mpl.gridspec.GridSpec(h+1, w, height_ratios=[10]*h + [3]) # pyright: ignore[reportAttributeAccessIssue]
    axes = [fig.add_subplot(gs[i // w, i % w]) for i in range(len(groups))]
    side_ax = fig.add_subplot(gs[h, :])  # bottommost row
    side_ax.axis("off")
    
    if "xlim" in kwargs and "ylim" in kwargs:
        area = (kwargs["xlim"][1] - kwargs["xlim"][0]) * (kwargs["ylim"][1] - kwargs["ylim"][0])
    else:
        umap_x = adata.obsm[umap_obsm or "X_umap"][:, 0]
        umap_y = adata.obsm[umap_obsm or "X_umap"][:, 1]
        area = (max(umap_x) - min(umap_x)) * (max(umap_y) - min(umap_y))
    
    s = 1000 / area

    is_groups = feature in adata.obs and pd.api.types.is_categorical_dtype(adata.obs[feature].dtype) # pyright: ignore[reportAttributeAccessIssue]

    if is_groups:
        # Build handles for custom legend
        categories = adata.obs[feature].cat.categories
        colors = adata.uns[feature + "_colors"]
        color_dict = dict(zip(categories, colors))
        if legend_order:
            categories = [c for c in legend_order if c in categories]
        else:
            categories = sorted(categories, key=int if feature == "cluster" else None)
        handles = [
            plt.Line2D([], [], marker='s', linestyle='None', color=color_dict[cat], label=cat, markersize=6) # pyright: ignore[reportPrivateImportUsage]
            for cat in categories
        ]
    else:
        # Compute shared vmin and vmax for continuous features
        data = adata.obs[feature] if feature in adata.obs.columns else adata.raw[:, feature].X
        if hasattr(data, "to_numpy"):  # pandas Series
            values = data.to_numpy().flatten()
        elif not hasattr(data, "toarray"):  # dense array
            values = data.flatten() 
        else:
            values = data.toarray().flatten()
        vmin = np.nanmin(values)
        vmax = np.nanmax(values)

        # Setup shared ScalarMappable for colorbar
        if vmin < 0 and vmax > 0:
            norm = mpl.colors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
            vcenter = 0
        else:
            norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax) # pyright: ignore[reportAttributeAccessIssue]
        sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm) # pyright: ignore[reportAttributeAccessIssue]
        sm.set_array([])

    # Create each subplot
    for i, t in enumerate(groups[:4]):
        print("test1")
        adata_sub = adata[adata.obs[group_key] == t]
        adata_sub.uns = adata.uns
        
        if not is_groups:
            kwargs = {"vmin": vmin, "vmax": vmax, "cmap": cmap} | kwargs
            if vcenter is not None:
                kwargs = {"vcenter": vcenter} | kwargs
        
        umap(
            adata_sub,
            feature=feature,
            ax=axes[i],
            side_ax=None,
            umap_obsm=umap_obsm,
            legend_loc="side",
            s=s,
            **kwargs,
        )
        axes[i].set_title(title(rename(t)))
        axes[i].set_xlabel(None) # pyright: ignore[reportArgumentType]
        axes[i].set_ylabel(None) # pyright: ignore[reportArgumentType]
        print("test2")

    # Add either colorbar or legend
    
    if is_groups:
        legend_kws = {
            "handles": handles,
            "loc": "upper center",
            "title": textwrap.fill(title(rename(feature)), 15),
            "fontsize": 10,
            "title_fontsize": 13,
            "cmap": cmap,
        } | legend_kws
        legend = side_ax.legend(**legend_kws)
        legend.get_title().set_multialignment('center')
    else:
        side_ax.axis("on")
        make_colorbar(sm, side_ax, f"{rename(feature)} Expression", vmin, vmax)
        
    plt.tight_layout()
    return fig

In [ ]:
types_by_supertype = adata.obs.groupby("cell_supertype")["cell_type"].unique().dropna().to_dict()
types_by_supertype = {k: set(v) for k, v in types_by_supertype.items()}

def get_types_by_supertypes(supertypes):
    result = set()
    for st in supertypes:
        result = result.union(types_by_supertype[st])
    return result

## General Figures and Cell Counts

### UMAP

In [ ]:
umap_type_order = [
    "AT1", "AT2 I", "AT2 II", "Club", "Goblet", "Ciliated", "Fibroblast", "Pericyte / VSMC",
    "Alveolar Macrophage", "Interstitial Macrophage", "Monocyte-derived Macrophage", "IFN-Activated Macrophage", "cDC1", "cDC2", "Migratory cDC", "pDC",
    "Treg", "Tfh", "Th17", "Naive CD8+ T", "Effector CD8+ T", "γδ T", "NK", "ILC2",
    "IgA+ Plasma", "IgG1+ Plasma", "IgG2b+ Plasma", "IgG2c+ Plasma", "IgM+ Plasma", "aCap", "gCap", "Erythroid",
    "Follicular B", "Activated B", "Circulating B", "Resident Memory B", "GC B", "Fcrl5+ Atypical B", "Neutrophil"
]
for c in adata.obs["cell_type"].unique():
    if c not in umap_type_order:
        print(c)

umap_split(
    adata, 
    "cell_type", 
    "sample", 
    figsize=(10, 10),
    legend_kws={"ncol": 5}, 
    s=2, 
    legend_order=umap_type_order
)
plt.savefig(os.path.join(output_dir, "figures", "cell_type_split.png"), bbox_inches="tight");

In [ ]:
umap_supertype_order = [
    "AT1", "AT2", "Club", 
    "Goblet", "Ciliated", "Endothelial",
    "Macrophage", "DC", "Neutrophil",
    "T", "NK", "ILC",
    "B", "Plasma", "Erythroid",
    "Fibroblast", "Pericyte / VSMC",
]
for c in adata.obs["cell_supertype"].unique():
    if c not in umap_supertype_order:
        print(c)

umap_split(
    adata, 
    "cell_supertype", 
    "sample", 
    figsize=(10, 10),
    legend_kws={"ncol": 6, "title": "Major Cell Type"}, 
    s=2, 
    legend_order=umap_supertype_order
)
plt.savefig(os.path.join(output_dir, "figures", "cell_supertype_split.png"), bbox_inches="tight");

### Cell Counts

In [ ]:
def calculate_cell_counts(anndata, sample_name="sample", type_columns=["cell_supertype", "cell_type"]):
    cell_counts = anndata.obs.groupby(["sample", *type_columns]).size()
    cell_counts = cell_counts[cell_counts.values != 0].reset_index(name="count")
    cell_counts = cell_counts.pivot_table(
        index="sample", 
        columns=type_columns, 
        values="count",
        fill_value=0
    ).loc[treatment_groups]
    cell_counts = pd.concat([cell_counts, pd.DataFrame([cell_counts.sum()], index=['Total'])])
    cell_counts = cell_counts.astype(int)
    return cell_counts

In [ ]:
cell_counts = calculate_cell_counts(adata)
cell_counts

In [ ]:
cell_counts.to_csv("../output/cell_counts.csv")

### Proportions

In [ ]:
def graph_proportions(df, colors, figsize=(10, 10), legend_title=None):
    fig = plt.figure(figsize=figsize, dpi=DPI)
    ax = plt.subplot(1, 1, 1)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    # df = df[list(set(df.columns) - {"B cell", "T/NK cell"})]
    df = df.div(df.sum(axis=1), axis=0) * 100
    
    bottom = np.zeros(len(df.index))
    for ct in df.columns:
        ax.bar(
            df.index,
            df[ct],
            bottom=bottom,
            label=ct,
            color = colors[ct]
        )

        # place text in the middle of the bar segments showing the value
        for i, x in enumerate(df.index):
            height = df.loc[x, ct]
            if height <= 0: # only display text for large enough segments
                continue
            ax.text(
                x, 
                bottom[i] + height/2,
                f"{height:.2f}%",
                ha="center", va="center",
                fontsize=10, color="black", weight="bold",
                # path_effects=[mpl.patheffects.withStroke(linewidth=0.5, foreground="white")]
            )
        
        bottom += df[ct]
    
    plt.xlim((-0.5, 1.5))
    plt.ylim((0, 100))
    plt.grid(False)
    plt.tight_layout()
    plt.ylabel("Percent of Cells")
    plt.xlabel(None)
    # plt.title("Cell Type Proportions")
    locs, labels = plt.xticks()
    for l in labels:
        print(l.get_text())
        l.set_text(title(rename(l.get_text())))
    plt.xticks(locs, labels, rotation=0)
    ax.tick_params(length=0)
    
    ax.legend()
    handles, labels = ax.get_legend_handles_labels()
    legend = ax.legend(
        handles[::-1], 
        [rename(l) for l in labels[::-1]], 
        bbox_to_anchor=(1, 0.5), 
        loc="center left", 
        title=legend_title,
        title_fontsize=14,
    )

In [ ]:
_colors = dict(zip(adata.obs["cell_supertype"].cat.categories, adata.uns["cell_supertype_colors"]))
_df = cell_counts.loc[["VEH", "HDM_DNA"]].groupby(level=0, axis=1).sum().drop([], axis=1)
_df = _df[supertype_order]
graph_proportions(_df, _colors, figsize=(3, 6), legend_title="Major Cell Type")
plt.savefig(os.path.join(output_dir, "figures", "cell_supertype_proportions.png"), bbox_inches="tight")

In [ ]:
_colors = dict(zip(adata.obs["cell_type"].cat.categories, adata.uns["cell_type_colors"]))
_df = cell_counts.loc[["VEH", "HDM_DNA"]].copy()
_df.columns = _df.columns.get_level_values(1)
graph_proportions(_df, _colors, figsize=(3, 12))
plt.savefig(os.path.join(output_dir, "figures", "cell_type_proportions.png"), bbox_inches="tight")

In [ ]:
_colors = dict(zip(adata.obs["cell_type"].cat.categories, adata.uns["cell_type_colors"]))
_df = cell_counts["B"].loc[["VEH", "HDM_DNA"]]
_df = _df[[c for c in type_order if c in _df.columns]]
graph_proportions(_df, _colors,  figsize=(3, 4))
plt.savefig(os.path.join(output_dir, "figures", "cell_proportions_b.png"), bbox_inches="tight")

In [ ]:
_colors = dict(zip(adata.obs["cell_type"].cat.categories, adata.uns["cell_type_colors"]))
_df = cell_counts["T"].loc[["VEH", "HDM_DNA"]]
_df = _df[[c for c in type_order if c in _df.columns]]
graph_proportions(_df, _colors,  figsize=(3, 4))
plt.savefig(os.path.join(output_dir, "figures", "cell_proportions_t.png"), bbox_inches="tight")

In [ ]:
_colors = dict(zip(adata.obs["cell_type"].cat.categories, adata.uns["cell_type_colors"]))
_df = cell_counts[["Macrophage", "DC"]].loc[["VEH", "HDM_DNA"]].copy()
_df.columns = _df.columns.droplevel(0)
_df = _df[[c for c in type_order if c in _df.columns]]

graph_proportions(_df, _colors,  figsize=(3, 4))
plt.savefig(os.path.join(output_dir, "figures", "cell_proportions_macrophage_dc.png"), bbox_inches="tight")

In [ ]:
_colors = dict(zip(adata.obs["cell_type"].cat.categories, adata.uns["cell_type_colors"]))
_df = cell_counts["Plasma"].loc[["VEH", "HDM_DNA"]]
_df = _df[[c for c in type_order if c in _df.columns]]
graph_proportions(_df, _colors,  figsize=(3, 4))
plt.savefig(os.path.join(output_dir, "figures", "cell_proportions_plasma.png"), bbox_inches="tight")

In [ ]:
cell_counts

In [ ]:
_colors = dict(zip(adata.obs["cell_type"].cat.categories, adata.uns["cell_type_colors"]))
_df = cell_counts[["AT1", "AT2", "Club", "Goblet", "Ciliated", "Pericyte / VSMC", "Fibroblast", "Endothelial", "Erythroid"]].loc[["VEH", "HDM_DNA"]]
_df.columns = _df.columns.get_level_values(1)
_df = _df[[c for c in type_order if c in _df.columns]]
graph_proportions(_df, _colors,  figsize=(3, 4))
plt.savefig(os.path.join(output_dir, "figures", "cell_proportions_non_immune.png"), bbox_inches="tight")

### Counts

In [ ]:
def graph_counts(df: pd.DataFrame, figsize=(10, 6), figure_title=None) -> mpl.axes.Axes:
    df = df.transpose()
    ax = df[[c for c in df.columns if c != "Total"]].plot.bar(figsize=figsize)
    ax.get_figure().set_dpi(DPI)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_axisbelow(True)
    
    plt.grid(axis="x")
    plt.ylabel("Number of Cells")
    plt.xlabel("Cell Type")
    plt.title(figure_title)
    locs, labels = plt.xticks()
    for l in labels:
        l.set_text(title(rename(l.get_text())))
    plt.xticks(locs, labels, rotation=90)
    # ax.tick_params(length=0)
    
    plt.legend()
    handles, labels = ax.get_legend_handles_labels()
    legend = ax.legend(
        handles, 
        [title(rename(l)) for l in labels], 
        title="Treatment",
        title_fontsize=14,
        facecolor="white",
        framealpha=1.0,
        frameon=True,
        borderpad=0.5,
    )
    
    return ax

In [ ]:
adata.uns["rename_dict"] |= {"VEH": "JetPEI", "HDM_DNA": "JetPEI + HDM DNA"}

In [ ]:
_df = cell_counts.groupby(level=0, axis=1).sum()
_df.loc["HDM_DNA"] / _df.loc["VEH"]

In [ ]:
_df = cell_counts.groupby(level=0, axis=1).sum()[supertype_order]
ax = graph_counts(_df.loc[["VEH", "HDM_DNA"]], figsize=(12, 6))
plt.yticks(range(0, 1601, 200))
txt = (
    r"$\bf{ Figure\ 1.\ Increased\ number\ of\ immune\ cells\ in\ lungs\ of\ mice\ treated\ with\ HDM\ DNA.}$" 
    "Mice were treated intranasally with either HDM DNA or Vehicle control (JetPEI alone) for nine weeks. " 
    "Lung cells were sequenced using the 10X Genomics Flex kit for single-nucleotide RNA sequencing. "
    "Cells were annotated with automated clustering and marker genes were compared against single-cell databases. "
    "In the HDM DNA group, there was a large increase in the number of neutrophils, plasma cells, "
    "B cells, and T cells, and a smaller increase in the number of dendritic cells and macrophages."
)
ax.figure.text(0.1, -0.17, txt, wrap=True, horizontalalignment='left', verticalalignment='top', fontsize=14)
# plt.savefig(os.path.join(output_dir, "figures", "cell_supertype_counts.png"), bbox_inches="tight")

In [ ]:
_df = cell_counts.groupby(level=1, axis=1).sum()[type_order]
graph_counts(_df)
plt.legend(loc="upper right", bbox_to_anchor=(0.9, 1.0))
plt.savefig(os.path.join(output_dir, "figures", "cell_type_counts.png"), bbox_inches="tight")

## Import data from GSVA in R

In [ ]:
import glob

file_list = glob.glob("../data/msigdb/msigdb*.csv")
dfs = [pd.read_csv(f, index_col=0) for f in file_list]
msigdb_combined_df = (
    pd.concat(dfs, ignore_index=True)
    .drop_duplicates(subset=["gene_symbol", "gs_name"])
)
geneset_sizes = msigdb_combined_df.groupby("gs_name").size()
geneset_names_and_sizes = {k: f"({v}) {k}" for k, v in geneset_sizes.items()}
geneset_genes = msigdb_combined_df.groupby("gs_name").agg(
    sizes = ("gene_symbol", "count"),
    genes = ("gene_symbol", lambda x: ",".join(x)),
)
print(len(geneset_names_and_sizes), len(geneset_genes))
msigdb_combined_df

In [ ]:
from tqdm import tqdm
comparisons = [("VEH", "HDM_DNA"), ("VEH", "HDM_Extract")]

r_gsva = {}
for cell_type in tqdm(cell_types):
    n = cell_type.replace(" ", "_").replace("/", "_").replace("ï", "i")
    # r_gsva_de[cell_type] = pd.read_csv(f"gsva_results/gsva_DE_{n}.csv", index_col=0)
    r_gsva[cell_type] = pd.read_csv(f"../output/gsva/gsva_{n}.csv", index_col=0)
    for comp in comparisons:
        r_gsva[cell_type][f"{comp[1]}_vs_{comp[0]}"] = r_gsva[cell_type][comp[1]] - r_gsva[cell_type][comp[0]]

In [ ]:
gsva_df = (
    pd.concat(r_gsva, names=["cell_type"])
    .reset_index(level=0)
    .rename(columns={"level_0": "cell_type"})
).join(
    msigdb_combined_df.groupby("gs_name")
    .agg(
        size = ("gene_symbol", "count"),
        genes = ("gene_symbol", lambda x: ",".join(x)),
    )
)

In [ ]:
gsva_df_filtered = gsva_df[
    (abs(gsva_df["HDM_DNA_vs_VEH"]) > 0.25)
    # | (abs(gsva_df["HDM_Extract_vs_VEH"]) > 0.5)
]
print(len(gsva_df), len(gsva_df_filtered))
gsva_df_filtered.to_excel(os.path.join(output_dir, "gsva_enrichment_DNA_vs_VEH.xlsx"))

In [ ]:
gsva_df

In [ ]:
def gsva_heatmap(gsva_df, pathways: dict, comparison: str, cell_type_order, label_percentile=10, **kwargs):
    pathway_keys, pathway_labels = zip(*pathways.items())
    pathway_keys, pathway_labels = list(pathway_keys), list(pathway_labels)
    display(pathway_keys, pathway_labels)
    _pathwaydf = gsva_df.loc[pathway_keys].pivot(columns="cell_type", values=comparison).reindex(pathway_keys)
    display(_pathwaydf)
    _pathwaydf = _pathwaydf[[c for c in cell_type_order if c in _pathwaydf.columns]]
    inverse_percentile = np.nanpercentile(np.abs(_pathwaydf.to_numpy().flatten()), 100 - label_percentile)
    # dendrogram_w = 1
    width_columns = len(_pathwaydf.columns) / 1.8
    width_text = max(_pathwaydf.index.str.len()) / 13
    height_rows = len(_pathwaydf.index) / 3.2
    height_text = max(_pathwaydf.columns.str.len()) / 9
    w = width_columns + width_text
    h = height_rows + height_text
    sc.set_figure_params(scanpy=True)
    cg = sns.clustermap(
        _pathwaydf,
        cmap="bwr",
        figsize=(w, h),
        dendrogram_ratio=(0, 0.01),
        row_cluster=False,
        col_cluster=False,
        vmin = -1,
        vmax = 1,
        xticklabels=True,
        yticklabels=True,
        annot=_pathwaydf.apply(
            lambda x: x.abs().round(2)
            .where(~x.abs().between(-0.01, inverse_percentile-0.01), "")
            .astype(str)
        ) if label_percentile > 0 else None,
        fmt="",
        **kwargs,
    )
    cg.ax_heatmap.set_xlabel(None)
    cg.ax_heatmap.grid(False)
    cg.ax_heatmap.set_xticks(
        cg.ax_heatmap.get_xticks(), 
        labels=cg.ax_heatmap.get_xticklabels(), 
        rotation=90
    )
    cg.ax_heatmap.set_yticklabels(pathway_labels)
    
    # Debug lines for relative widths/heights of text and heatmap grid
    # factor = 0.85
    # line = mpl.lines.Line2D((0, width_columns/w*factor), (0, 0), lw=5., color='r', alpha=1) # pyright: ignore[reportAttributeAccessIssue]
    # cg.figure.add_artist(line)
    # line = mpl.lines.Line2D((width_columns/w*factor, factor), (0, 0), lw=5., color='b', alpha=1) # pyright: ignore[reportAttributeAccessIssue]
    # cg.figure.add_artist(line)
    
    # factor = 1.0
    # line = mpl.lines.Line2D((0, 0), (0, height_text/h*factor), lw=5., color='b', alpha=1) # pyright: ignore[reportAttributeAccessIssue]
    # cg.figure.add_artist(line)
    # line = mpl.lines.Line2D((0, 0), (height_text/h*factor, factor), lw=5., color='r', alpha=1) # pyright: ignore[reportAttributeAccessIssue]
    # cg.figure.add_artist(line)

In [ ]:
all_present_genes = set(adata.var_names)

In [ ]:
import re
search = re.compile(r"AIM2")
for p in geneset_sizes.index.unique():
    if search.search(p):
        genes = set(geneset_genes.loc[p].values[1].split(","))
        present = genes.intersection(all_present_genes)
        missing = genes - all_present_genes
        print(">", p in gsva_df.index, " \t", p)#, geneset_genes.loc[p].values)
        print(f"{len(present)} genes present, {len(missing)} genes missing")

In [ ]:
immunosuppression_pathways = {
    "CD8": [
        "REACTOME_CO_INHIBITION_BY_PD_1",
        "GSE9650_NAIVE_VS_EXHAUSTED_CD8_TCELL_UP",
        "GSE9650_NAIVE_VS_EXHAUSTED_CD8_TCELL_DN",
        "GSE41867_MEMORY_VS_EXHAUSTED_CD8_TCELL_DAY30_LCMV_UP",
        "GSE41867_MEMORY_VS_EXHAUSTED_CD8_TCELL_DAY30_LCMV_DN",
    ],
    
    "Macrophage": [
        "REACTOME_INTERLEUKIN_4_AND_INTERLEUKIN_13_SIGNALING",
        "GSE5099_CLASSICAL_M1_VS_ALTERNATIVE_M2_MACROPHAGE_UP",
        "GSE5099_CLASSICAL_M1_VS_ALTERNATIVE_M2_MACROPHAGE_DN",
        "GSE5099_DAY3_VS_DAY7_MCSF_TREATED_MACROPHAGE_UP",
        "GSE5099_DAY3_VS_DAY7_MCSF_TREATED_MACROPHAGE_DN",
    ]
}

gsva_heatmap(
    gsva_df.loc[gsva_df["cell_type"].isin(["Naive CD8+ T", "Effector CD8+ T"])], 
    {k: k for k in immunosuppression_pathways["CD8"]}, 
    "HDM_DNA_vs_VEH",
    type_order,
    label_percentile = 100,
    cbar_pos = (1.0, 0, 0.4, 0.03),
    cbar_kws = {"orientation": "horizontal"},
)
plt.suptitle("CD8 Exhaustion Genesets, DNA minus VEH", backgroundcolor="white", va="bottom", ha="center", fontsize=18, x=0.7)

macrophage_types = adata[adata.obs["cell_supertype"] == "Macrophage"].obs["cell_type"].unique()
gsva_heatmap(
    gsva_df.loc[gsva_df["cell_type"].isin(macrophage_types)], 
    {k: k for k in immunosuppression_pathways["Macrophage"]}, 
    "HDM_DNA_vs_VEH",
    type_order,
    label_percentile = 100,
    cbar_pos = (1.0, 0, 0.4, 0.03),
    cbar_kws = {"orientation": "horizontal"},
)
plt.suptitle("Macrophage M2 Genesets, DNA minus VEH", backgroundcolor="white", va="bottom", ha="center", fontsize=18, x=0.7)

In [ ]:
dna_sensing_pathways = {
    "GOBP_CGAS_STING_SIGNALING_PATHWAY": "cGAS-STING Pathway",
    "REACTOME_STING_MEDIATED_INDUCTION_OF_HOST_IMMUNE_RESPONSES": "STING-mediated Immune Responses",
    # "GOBP_POSITIVE_REGULATION_OF_CGAS_STING_SIGNALING_PATHWAY": "Positive Regulation of cGAS-STING",
    "GOBP_NEGATIVE_REGULATION_OF_CGAS_STING_SIGNALING_PATHWAY": "Negative Regulation of cGAS-STING",
    "HALLMARK_INTERFERON_ALPHA_RESPONSE": "Interferon Alpha Response",
    # "GOBP_AIM2_INFLAMMASOME_COMPLEX": "AIM2 Inflammasome Complex",
    # "GOBP_REGULATION_OF_AIM2_INFLAMMASOME_COMPLEX_ASSEMBLY": "Regulation of AIM2 Inflammasome Assembly",
    "REACTOME_INFLAMMASOMES": "Inflammasomes",
    "REACTOME_TOLL_LIKE_RECEPTOR_9_TLR9_CASCADE": "TLR9 Cascade",
    # "REACTOME_TRAF6_MEDIATED_INDUCTION_OF_NFKB_AND_MAP_KINASES_UPON_TLR7_8_OR_9_ACTIVATION": "TLR7/8/9 -> TRAF6 -> NFkb / MAPK",
    # "KEGG_TOLL_LIKE_RECEPTOR_SIGNALING_PATHWAY": "General TLR Pathways",
    # "KEGG_MEDICUS_REFERENCE_TLR7_9_IRF7_SIGNALING_PATHWAY": "TLR7/9 -> IRF7 Pathway",
    "KEGG_MEDICUS_REFERENCE_TLR7_8_9_IRF5_SIGNALING_PATHWAY": "TLR7/8/9 -> IRF5 Pathway",
    "REACTOME_TRAF6_MEDIATED_IRF7_ACTIVATION_IN_TLR7_8_OR_9_SIGNALING": "TRAF6-Mediated TLR7/8/9 -> IRF7",
    "GOBP_MYD88_DEPENDENT_TOLL_LIKE_RECEPTOR_SIGNALING_PATHWAY": "MYD88-Dependent TLR Pathway",
}
gsva_heatmap(
    gsva_df, 
    dna_sensing_pathways, 
    "HDM_DNA_vs_VEH", 
    type_order,
    cbar_pos = (0.72, 0.1, 0.1, 0.03),
    cbar_kws = {"orientation": "horizontal"},
)
plt.suptitle("DNA Sensing Pathways, DNA minus VEH", backgroundcolor="white", va="bottom", ha="center", fontsize=18, x=0.37)

In [ ]:
features = "Adar,B2m,Batf2,Bst2,C1s1,Casp1,Casp8,Ccrl2,Cd47,Cd74,Cmpk2,Cmtr1,Cnp,Csf1,Cxcl10,Cxcl11,Ddx60,Dhx58,Eif2ak2,Elf1,Epsti1,Gbp2,Gbp3,Gmpr,Helz2,Herc6,H2-Q10,H2-Q7,Ifi27,Ifi27l2b,Ifi30,Ifi35,Ifi44,Ifi44l,Ifih1,Ifit2,Ifit3,Ifitm1,Ifitm2,Ifitm3,Il15,Il4ra,Il7,Irf1,Irf2,Irf7,Irf9,Isg15,Isg20,Lamp3,Lap3,Lgals3bp,Lpar6,Ly6e,Mov10,Mvb12a,Mx2,Ncoa7,Nmi,Nub1,Oas1a,Oas1g,Oasl1,Ogfr,Parp12,Parp14,Parp9,Plscr1,Pnpt1,Procr,Psma3,Psmb8,Psmb9,Psme1,Psme2,Ripk2,Rnf31,Rsad2,Rtp4,Samd9l,Sell,Slc25a28,Sp110,Stat2,Tap1,Tdrd7,Tent5a,Tmem140,Trafd1,Trim14,Trim21,Trim25,Trim26,Trim12c,Trim5,Txnip,Uba7,Ube2l6,Usp18,Wars1"
features = features.split(",")
samples = ["VEH", "HDM_DNA"]
for s in samples:
    data = adata[adata.obs["sample"] == s, feature].X.toarray().flatten()
    print(f"{feature} in {s} mean: {np.mean(data):.2f}, pct_expressing: {np.mean(data > 0)*100:.2f}")

In [ ]:
_pathways = {
    "General": {
        "GOBP_NEGATIVE_REGULATION_OF_IMMUNE_RESPONSE": "GOBP_NEGATIVE_REGULATION_IMMUNE_RESPONSE"
    },
    "Cytokine-driven Immunosuppression": {
        "HALLMARK_IL6_JAK_STAT3_SIGNALING": "HMK_IL6_JAK_STAT3",
        "HALLMARK_TGF_BETA_SIGNALING": "HMK_TGF_BETA",
        "HALLMARK_IL2_STAT5_SIGNALING": "HMK_IL2_STAT5",
        "REACTOME_TNF_RECEPTOR_SUPERFAMILY_TNFSF_MEMBERS_MEDIATING_NON_CANONICAL_NF_KB_PATHWAY": "RME_TNF_NFKB_NON_CANONICAL",
        "KEGG_MEDICUS_REFERENCE_CYTOKINE_JAK_STAT_SIGNALING_PATHWAY": "KGG_CYTOKINE_JAK_STAT",
        "REACTOME_CYTOKINE_SIGNALING_IN_IMMUNE_SYSTEM": "RME_CYTOKINE_SIGNALING",
        "REACTOME_INTERLEUKIN_10_SIGNALING": "RME_IL10_SIGNALING"
    },
    "Myeloid suppressive M2-like": {
        "HALLMARK_TNFA_SIGNALING_VIA_NFKB": "HMK_TNFA_SIGNALING_VIA_NFKB",
        "REACTOME_TOLL_LIKE_RECEPTOR_CASCADES": "RME_TOLL_LIKE_RECEPTOR_CASCADES",
        "REACTOME_CHEMOKINE_RECEPTORS_BIND_CHEMOKINES": "RME_CHEMOKINE_RECEPTORS_BIND_CHEMOKINES"
    },
    "T cell exhaustion / checkpoint-mediated supression": {
        "HALLMARK_INTERFERON_ALPHA_RESPONSE": "HMK_IFNA_RESPONSE",
        "HALLMARK_INTERFERON_GAMMA_RESPONSE": "HMK_IFNG_REPSONSE",
        "REACTOME_CO_INHIBITION_BY_CTLA4": "RME_CO_INHIBITION_BY_CTLA4"
    },
    "Metabolic Immunosuppression": {
        "HALLMARK_HYPOXIA": "HMK_HYPOXIA",
        "HALLMARK_GLYCOLYSIS": "HMK_GLYCOLYSIS",
        "HALLMARK_OXIDATIVE_PHOSPHORYLATION": "HMK_OX_PHOS",
        "HALLMARK_FATTY_ACID_METABOLISM": "HMK_FATTY_ACID_METABOLISM",
        "REACTOME_FATTY_ACID_METABOLISM": "RME_FATTY_ACID_METABOLISM",
        "REACTOME_MITOCHONDRIAL_FATTY_ACID_BETA_OXIDATION": "RME_MITO_FATTY_ACID_BETA_OXIDATION",
        "REACTOME_GLUTATHIONE_SYNTHESIS_AND_RECYCLING": "RME_GLUTATHIONE_SYNTHESIS_AND_RECYCLING",
        "HALLMARK_REACTIVE_OXYGEN_SPECIES_PATHWAY": "HMK_ROS_PATHWAY"
    },
    "CAF / ECM Remodeling": {
        "REACTOME_EXTRACELLULAR_MATRIX_ORGANIZATION": "RME_ECM_ORGANIZATION",
        "REACTOME_COLLAGEN_FORMATION": "RME_COLLAGEN_FORMATION",
        "HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION": "HMK_EMT",
        "REACTOME_INTEGRIN_SIGNALING": "RME_INTEGRIN_SIGNALING"
    },
    "Antigen Loss or Distortion": {
        "REACTOME_CLASS_I_MHC_MEDIATED_ANTIGEN_PROCESSING_PRESENTATION": "RME_MHCI_ANTIGEN_PROCESSING_PRESENTATION",
        "REACTOME_ANTIGEN_PRESENTATION_FOLDING_ASSEMBLY_AND_PEPTIDE_LOADING_OF_CLASS_I_MHC": "RME_MHCI_FOLDING_ASSEMBLY_PEPTIDE_LOADING",
        "REACTOME_NEF_MEDIATED_DOWNREGULATION_OF_MHC_CLASS_I_COMPLEX_CELL_SURFACE_EXPRESSION": "RME_NEF_MEDIATED_DOWNREGULATION_MHCI_SURFACE_EXPRESSION",
        "REACTOME_MHC_CLASS_II_ANTIGEN_PRESENTATION": "RME_MHCII_ANTIGEN_PRESENTATION"
    }
}

_pathways_flat = [k for group in _pathways.values() for k in group]
_df = gsva_df.loc[_pathways_flat]
_df = _df.pivot(columns="cell_type", values="HDM_DNA_vs_VEH")[type_order]
_df = _df.loc[_pathways_flat].fillna(0)
cg = sns.clustermap(
    _df,
    cmap="bwr",
    center=0,
    row_cluster=False,
    col_cluster=False,
    figsize=(24, 12),
    dendrogram_ratio=(0, 0.2),
    colors_ratio=(0, 0),
    cbar_pos=(0.57, 0.05, 0.15, 0.015),
    cbar_kws={"orientation": "horizontal"},
    annot=_df.apply(
        lambda x: x.abs().round(2)
        .where(~x.between(-0.495, 0.495), "")
        .astype(str)
    ),
    fmt="",
    annot_kws={"fontsize": 10, "color": "white"},
)

# Add category labels on the left
y = 0
for category, group in _pathways.items():
    n = len(group)
    cg.ax_heatmap.plot(
        (-0.2, -0.2), (y+0.2, y+n-0.2),
        # transform = cg.ax_heatmap.get_yaxis_transform(),
        lw = 2,
        color = "black",
        clip_on=False,
    )
    cg.ax_heatmap.text(-0.4, y + n/2, wrap_join(category.split(" "), width=20), ha="right", va="center")
    y += n

# modify the pathway labels on the right
labels = []
for group in _pathways.values():
    for v in group.values():
        labels.append(v)
    # p = p.split("_")
    # p = p[0][0] + p[0][-2] + p[0][-1] + "_" + "_".join(p[1:])
    # if len(p) > 45:
    #     p = f"{p[:20]}...{p[-20:]}"
    # labels.append(p)

# cg.ax_heatmap.tick_params(length=0)
cg.ax_heatmap.set_yticklabels(labels)
cg.ax_heatmap.set_xlabel("")
_title = cg.ax_heatmap.set_title("TME Immunosuppresion, HDM DNA minus VEH", backgroundcolor="white", va="bottom", fontsize=18)
cg.ax_heatmap.grid(False)
if cg.ax_cbar:
    cg.ax_cbar.set_xticks([_df.min(axis=None), -0.5, 0, 0.5, _df.max(axis=None)])
plt.tight_layout()

## Module Scores

In [ ]:
all_genes = set(adata.var_names)
for gene_start in ["Slamf"]:
    for gene in all_genes:
        if gene.startswith(gene_start):
            print(gene)

In [ ]:
print(", ".join(["Tox", "Nr4a1", "Nr4a2", "Nr4a3",
    "Batf", "Irf4", "Eomes",
    "Prdm1", "Ikzf2"]))

In [ ]:
immunosuppression_modules = {
    # 1) Exhaustion / dysfunction (CD8/Tex)
    "Tex_Checkpoints": [
        "Pdcd1", "Ctla4", "Havcr2", "Lag3", "Tigit",
        "Entpd1", "Cd244", "Tox2"
    ],
    "Tex_TF_program": [
        "Tox", "Nr4a1", "Nr4a2", "Nr4a3",
        "Batf", "Irf4", "Eomes",
        "Prdm1", "Ikzf2"  # optional included
    ],

    # 2) Cytotoxic effector
    "Cytotoxicity": [
        "Gzmb", "Gzmk", "Prf1", "Nkg7",
        "Ifng", "Ccl5", "Fasl"
    ],

    # 3) Treg / immunosuppressive T cells
    "Treg": [
        "Foxp3", "Il2ra", "Ctla4", "Ikzf2",
        "Tnfrsf4", "Tnfrsf18", "Tgfb1"
    ],

    # 4) M2-like TAM polarization
    "M2_AltActivation": [
        "Mrc1", "Arg1", "Chil3", "Retnla",
        "Il10", "Ccl17", "Ccl22"
    ],
    "TAM_TumorAssociated": [
        "Apoe", "Lgals3", "Trem2", "Cst7",
        "Spp1", "Fabp5", "Itgax",
        "Mmp12", "Cd9"  # optional included
    ],

    # 5) Myeloid immunosuppression
    "Myeloid_Suppressive": [
        "Il10", "Tgfb1", "Arg1", "Nos2",
        "Cd274",        # PD-L1
        "Pdcd1lg2",     # PD-L2
        "Ido1", "Socs3"
    ]
}


for module_name, module in immunosuppression_modules.items():
    count = 0
    for gene in module:
        if gene not in all_genes:
            print("Missing", gene)
        else:
            count += 1
    print(f"Found {count}/{len(module)} genes in module '{module_name}'")


In [ ]:
comparison = adata[adata.obs["sample"].isin(["VEH", "HDM_DNA"])]
all_t = comparison[comparison.obs["cell_supertype"] == "T"].copy()
cd8_t = all_t[all_t.obs["cell_type"].isin(["Effector CD8+ T", "Naive CD8+ T"])].copy()
macrophage = comparison[comparison.obs["cell_supertype"] == "Macrophage"].copy()
myeloid = comparison[comparison.obs["cell_supertype"].isin(["Macrophage", "DC"])].copy()

sc.tl.score_genes(cd8_t, immunosuppression_modules["Tex_Checkpoints"], score_name="Score_Tex_Checkpoints")
sc.tl.score_genes(cd8_t, immunosuppression_modules["Tex_TF_program"], score_name="Score_Tex_TF_program")
sc.tl.score_genes(cd8_t, immunosuppression_modules["Cytotoxicity"], score_name="Score_Cytotoxicity")
sc.tl.score_genes(all_t, immunosuppression_modules["Treg"], score_name="Score_Treg")
sc.tl.score_genes(macrophage, immunosuppression_modules["M2_AltActivation"], score_name="Score_M2_AltActivation")
sc.tl.score_genes(macrophage, immunosuppression_modules["TAM_TumorAssociated"], score_name="Score_TAM_TumorAssociated")
sc.tl.score_genes(myeloid, immunosuppression_modules["Myeloid_Suppressive"], score_name="Myeloid_Suppressive")

In [ ]:
subset = adata[
    (adata.obs["sample"].isin(["VEH", "HDM_DNA"]))
    & (adata.obs["cell_supertype"] == "T")
]
umap_split(subset, feature="Score_Tex_Checkpoints", group_key="sample", umap_obsm="X_umap_t_cell");
sc.pl.embedding(subset, basis="X_umap_t_cell", color="cell_type")

In [ ]:
def violin(x=None, y=None, hue=None, xlabel=None, ylabel=None, rotate_xticks=False, order=None, title=None, rename_dict=None):
    fig, ax = plt.subplots(dpi=DPI)

    import scipy

    # 1. Map to unique group IDs (same as before)
    unique_x, labels_x = np.unique(x, return_inverse=True)
    unique_hue, labels_hue = np.unique(hue, return_inverse=True)

    combined_labels = np.vstack((labels_x, labels_hue)).T
    unique_combos, group_indices = np.unique(combined_labels, axis=0, return_inverse=True)

    # 2. Sort values by group_indices so we can split them
    sort_idx = np.argsort(group_indices)
    sorted_groups = group_indices[sort_idx]
    sorted_values = y[sort_idx]

    # 3. Find the split points and split the values array
    splits = np.where(np.diff(sorted_groups) != 0)[0] + 1
    grouped_values = np.split(sorted_values, splits)

    # 4. Calculate everything
    results = []
    for i, group_data in enumerate(grouped_values):
        combo = unique_combos[i]
        results.append({
            'x': unique_x[combo[0]],
            'hue': unique_hue[combo[1]],
            'mean': np.mean(group_data),
            'median': np.median(group_data),
            'mode': scipy.stats.mode(group_data, keepdims=True).mode[0],
            'positive_pct': np.sum(group_data > 0) / len(group_data) * 100,
            'std': np.std(group_data),
            'count': len(group_data)
        })
    
    for r in results:
        print(f"{r["x"]} ({r["hue"]})")
        print("Mean:", r["mean"])
        print("Median:", r["median"])
        print("Stdev:", r["std"])
        print("% Above 0:", r["positive_pct"])
        print()

    # x2 = [x + " " + hue for hue, x in zip(hue, x)]

    # Jittered points (doesn't work with hue)
    # sns.stripplot(
    #     x=x2,
    #     y=y,
    #     hue=hue,
    #     ax=ax,
    #     jitter=0.4,
    #     size=1,            # Point size
    #     color="black",      # Point color
    #     alpha=0.2,          # Optional transparency
    # )
    
    # Violin plot (no inner points)
    sns.violinplot(
        x=x,
        y=y,
        hue=hue,
        ax=ax,
        inner=None,
        order=order,
        width=0.9
    )
    
    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), frameon=False)
    
    if title:
        plt.title(rename(title))
    if xlabel:
        plt.xlabel(rename(xlabel))
    if ylabel:
        plt.ylabel(rename(ylabel))
    ax.xaxis.set_major_locator(mpl.ticker.FixedLocator(ax.get_xticks()))
    ax.set_xticklabels([rename(l.get_text()) for l in ax.get_xticklabels()])
    if rotate_xticks:
        plt.xticks(rotation=90, ha="right", va="center", rotation_mode="anchor")
    
    # plt.tight_layout()
    
    return results, fig

In [ ]:
def draw_module(adata, module_name, x="cell_type", hue="sample"):
    print(f"---{module_name}---")
    data, fig = violin(x=adata.obs[x], y=adata.obs["Score_" + module_name], hue=adata.obs[hue], rotate_xticks=True)
    ax = fig.axes[0]
    ax.set_title(ax.get_ylabel())
    ax.set_ylabel(None)
    ax.set_xlabel(None)
    return data

In [ ]:
module_data = {}
module_data["Tex_Checkpoints"] = draw_module(cd8_t, "Tex_Checkpoints")
module_data["Tex_TF_program"] = draw_module(cd8_t, "Tex_TF_program")
module_data["Cytotoxicity"] = draw_module(cd8_t, "Cytotoxicity")
module_data["Treg"] = draw_module(all_t, "Treg", x="cell_type")
module_data["M2_AltActivation"] = draw_module(macrophage, "M2_AltActivation")
module_data["TAM_TumorAssociated"] = draw_module(macrophage, "TAM_TumorAssociated")
module_data["Myeloid_Suppressive"] = draw_module(myeloid, "Myeloid_Suppressive")

In [ ]:
module_dataframe = None
for module_name, data in module_data.items():
    df = pd.DataFrame(data)
    df["module"] = module_name
    df = df.rename(columns={"x": "cell_type", "hue": "treatment"}).set_index(["module", "cell_type", "treatment"])
    if module_dataframe is None:
        module_dataframe = df
    else:
        module_dataframe = pd.concat([module_dataframe, df])
if module_dataframe is not None:
    module_dataframe.reset_index().to_excel(os.path.join(output_dir, "modules_immunosuppression.xlsx"), index=False)
module_dataframe

## Import data from Cellchat in R

In [ ]:
cellchat_dna = pd.read_csv(os.path.join(output_dir, "cellchat", "LR_communications_DNA.csv"))
print(len(cellchat_dna))
(-1/np.log(cellchat_dna.groupby("pathway_name")["prob"].sum())).loc["APP"]#.sort_values(ascending=False).head()